# Automatic Multi-Band Dispersion Overlay (Top 3 Bands)

This notebook automatically:
1. detects passbands from DOS,
2. extracts the top 3 bands,
3. plots Band 1 / Band 2 / Band 3 overlays for each MAT file.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path('.').resolve()))

from pinn_dispersion_from_mat import (
    load_pinn_results,
    resample_to_uniform,
    compute_spectrum,
    linear_dispersion,
    compute_dos,
    detect_band_gaps,
    extract_ridge_in_band,
)

In [ ]:
# ---------------- USER SETTINGS ----------------
mat_files = [
    '../Result/pinn_data.mat',
    # '../Result/new_data.mat',
]

if len(mat_files) == 1 and mat_files[0] == '../Result/pinn_data.mat':
    all_mats = sorted(Path('../Result').glob('*.mat'))
    if len(all_mats) > 1:
        mat_files = [str(p) for p in all_mats]

k_coupling = 100.0      # update to your model
mx = 1.0                # update to your model
n_harmonic = 4
pts_per_cycle = 12
skip_transient = 0.20
omega_min = 0.01
n_bands = 3
force_three_bands = False  # True => always split into exactly n_bands windows

# DOS band detection controls
dip_threshold_dB = 5.0
min_distance = 5

print('MAT files to process:')
for f in mat_files:
    print('  -', f)

In [ ]:
def top_passband_windows(omega_pos, spectrum, n_bands=3, omega_min=0.01,
                        dip_threshold_dB=5.0, min_distance=5, force_n_bands=False):
    """
    Detect passband peaks from DOS and return up to n_bands windows:
    [(w_lo, w_hi), ...], plus selected peak frequencies.
    """
    i0 = np.searchsorted(omega_pos, omega_min)
    omega_clip = omega_pos[i0:]
    spec_clip = spectrum[:, i0:]

    dos, dos_dB = compute_dos(spec_clip)
    gap_freqs, band_freqs = detect_band_gaps(
        omega_clip, dos_dB,
        dip_threshold_dB=dip_threshold_dB,
        min_distance=min_distance
    )

    if len(band_freqs) == 0:
        if force_n_bands:
            edges = np.linspace(omega_clip[0], omega_clip[-1], n_bands + 1)
            windows = [(float(edges[i]), float(edges[i + 1])) for i in range(n_bands)]
            peak_freqs = 0.5 * (edges[:-1] + edges[1:])
            return windows, peak_freqs, omega_clip, dos_dB
        # Fallback: one wide band over full clipped range
        return [(omega_clip[0], omega_clip[-1])], np.array([(omega_clip[0] + omega_clip[-1]) * 0.5]), omega_clip, dos_dB

    # Rank passband peaks by DOS magnitude and keep top n_bands
    band_idx = np.searchsorted(omega_clip, band_freqs)
    band_idx = np.clip(band_idx, 0, len(dos) - 1)
    peak_power = dos[band_idx]
    keep = np.argsort(peak_power)[::-1][:n_bands]
    peak_freqs = np.sort(band_freqs[keep])

    if force_n_bands and len(peak_freqs) < n_bands:
        # Force exactly n_bands by uniform frequency partition (plotting convenience mode).
        edges = np.linspace(omega_clip[0], omega_clip[-1], n_bands + 1)
        windows = [(float(edges[i]), float(edges[i + 1])) for i in range(n_bands)]
        peak_freqs = 0.5 * (edges[:-1] + edges[1:])
        return windows, peak_freqs, omega_clip, dos_dB

    # Build windows by midpoints between selected peaks
    mids = 0.5 * (peak_freqs[:-1] + peak_freqs[1:]) if len(peak_freqs) > 1 else np.array([])

    windows = []
    for i, w0 in enumerate(peak_freqs):
        lo = omega_clip[0] if i == 0 else mids[i - 1]
        hi = omega_clip[-1] if i == len(peak_freqs) - 1 else mids[i]

        # Refine with nearest detected gaps if available
        left_gaps = gap_freqs[gap_freqs < w0]
        right_gaps = gap_freqs[gap_freqs > w0]
        if len(left_gaps) > 0:
            lo = max(lo, left_gaps[-1])
        if len(right_gaps) > 0:
            hi = min(hi, right_gaps[0])

        if hi <= lo:
            # Safety fallback around peak
            bw = max((omega_clip[-1] - omega_clip[0]) / (4 * max(1, len(peak_freqs))), 1e-6)
            lo = max(omega_clip[0], w0 - bw)
            hi = min(omega_clip[-1], w0 + bw)

        windows.append((float(lo), float(hi)))

    return windows, peak_freqs, omega_clip, dos_dB


In [ ]:
# Process each MAT file and extract top-n bands
results = []
omega_max_lin = linear_dispersion(np.pi, k_coupling, mx)

for fp in mat_files:
    t_total, x_PINN_total, params = load_pinn_results(fp)
    X_raw = x_PINN_total.T  # (N_time, n_dof) -> (n_dof, M)

    t_u, X_u, dt, omega_nyq = resample_to_uniform(
        t_total, X_raw, omega_max_lin,
        n_harmonic=n_harmonic, pts_per_cycle=pts_per_cycle
    )

    k_pos, omega_pos, spectrum = compute_spectrum(
        t_u, X_u, skip_transient=skip_transient
    )

    windows, peak_freqs, omega_clip, dos_dB = top_passband_windows(
        omega_pos, spectrum,
        n_bands=n_bands,
        omega_min=omega_min,
        dip_threshold_dB=dip_threshold_dB,
        min_distance=min_distance,
        force_n_bands=force_three_bands
    )

    band_curves = []
    for (w_lo, w_hi) in windows:
        i_lo = np.searchsorted(omega_pos, w_lo)
        i_hi = np.searchsorted(omega_pos, w_hi)
        if i_hi <= i_lo + 1:
            band_curves.append(np.full(len(k_pos), np.nan))
            continue

        omega_band = omega_pos[i_lo:i_hi]
        spec_band = spectrum[:, i_lo:i_hi]
        ridge = extract_ridge_in_band(k_pos, omega_band, spec_band, omega_min=omega_band[0])
        band_curves.append(ridge)

    label = Path(fp).stem
    if 'phi2' in params:
        label += f" (phi2={params['phi2']:.3g})"

    results.append({
        'file': fp,
        'label': label,
        'k': k_pos,
        'bands': band_curves,
        'windows': windows,
        'peak_freqs': peak_freqs,
    })

print(f'Processed {len(results)} file(s).')
for r in results:
    print(f"\n{r['label']}")
    for bi, w in enumerate(r['windows'], start=1):
        print(f"  Band {bi}: ω in [{w[0]:.4f}, {w[1]:.4f}]")


In [ ]:
# Plot all 3 bands for all MAT files in one figure
plt.figure(figsize=(10, 6.5))

for r in results:
    for b, omega_b in enumerate(r['bands'], start=1):
        valid = np.isfinite(omega_b)
        if not np.any(valid):
            continue
        plt.plot(
            r['k'][valid] / np.pi,
            omega_b[valid],
            linewidth=2,
            label=f"{r['label']} - Band {b}"
        )

# Optional linear reference
k_line = np.linspace(0, np.pi, 300)
plt.plot(
    k_line / np.pi,
    linear_dispersion(k_line, k_coupling, mx),
    'k--',
    linewidth=1.8,
    alpha=0.75,
    label='linear reference'
)

plt.xlabel(r'Wavenumber $k/\pi$')
plt.ylabel(r'Frequency $\omega$ (rad/s)')
plt.title('Top-3 Bands Overlay for All MAT Files (Single Plot)')
plt.grid(alpha=0.25)
plt.legend(loc='best', fontsize=9, ncol=2)
plt.tight_layout()
plt.show()